This file will be used to find reviews, price discounts, update history over time

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
from typing import Optional, Dict, List
import csv
import random
from datetime import datetime, timezone

In [4]:
def find_appid(game_name: str, lookup_df: pd.DataFrame) -> Optional[int]:
    """
    Find AppID from the lookup file by matching game name
    """
    game_name_clean = game_name.strip()
    
    # Try exact match first (case insensitive)
    exact_match = lookup_df[lookup_df['name'].str.lower() == game_name_clean.lower()]
    if not exact_match.empty:
        return int(exact_match.iloc[0]['appid'])
    
    # Try partial match
    partial_match = lookup_df[lookup_df['name'].str.contains(game_name_clean, case=False, na=False, regex=False)]
    if not partial_match.empty:
        # Prefer closer matches
        for idx, row in partial_match.iterrows():
            if game_name_clean.lower() in row['name'].lower():
                return int(row['appid'])
        return int(partial_match.iloc[0]['appid'])
    
    return None

In [5]:
def getRawFeatures(app_id):
    url = f"https://store.steampowered.com/appreviewhistogram/{app_id}"
    response = requests.get(url).json()
    return response['results']['rollups']
df = pd.DataFrame(getRawFeatures(730))
df['date'] = pd.to_datetime(df['date'], unit = 's')
df[0:20]

,date,recommendations_up,recommendations_down
0,2012-05-01,1,0
1,2012-08-01,1630,208
2,2012-09-01,1003,88
3,2012-10-01,511,59
4,2012-11-01,591,62
5,2012-12-01,867,93
6,2013-01-01,690,78
7,2013-02-01,598,68
8,2013-03-01,529,55
9,2013-04-01,495,53


In [6]:
results = []
lookup_df = pd.read_csv("complete_steam_lookup_2026.csv")
with open('game_list.csv', mode = 'r', newline = '') as file:
    reader = csv.reader(file)
    header = next(reader)
    for row in reader:
        app_id = find_appid(row[0], lookup_df)
        df_details = getRawFeatures(app_id)

        df = pd.DataFrame(df_details)
        if df is not None:
            df['date'] = pd.to_datetime(df['date'], unit = 's')
            df['Game'] = row[0]
        results.append(df)
        time.sleep(random.uniform(2, 5))
full_df = pd.concat(results, axis = 0, ignore_index = True)
full_df

,date,recommendations_up,recommendations_down,Game
0,2012-05-01,1,0,Counter-Strike 2
1,2012-08-01,1630,208,Counter-Strike 2
2,2012-09-01,1003,88,Counter-Strike 2
3,2012-10-01,511,59,Counter-Strike 2
4,2012-11-01,591,62,Counter-Strike 2
...,...,...,...,...
4951,2026-01-01,247,105,Black Desert
4952,2026-02-01,176,94,Black Desert
4953,2026-03-01,354,157,Black Desert
4954,2026-04-01,255,125,Black Desert


In [8]:
full_df['Percent_Positive'] = np.log(1+full_df['recommendations_up']+full_df['recommendations_down']) * full_df['recommendations_up']/(full_df['recommendations_down']+ full_df['recommendations_up'])
full_df.to_csv('Data/Reviews.csv')

In [ ]:
def get_updates(appid, max_window=160):
    base_url = "https://api.steampowered.com/ISteamNews/GetNewsForApp/v0002/"
    current_unix = 1780272000 
    seconds_in_month = 30 * 24 * 60 * 60 
    
    mapped_data = []
    last_total_count = None

    for i in range(max_window + 1):
        target_unix = current_unix - (i * seconds_in_month)
        
        dt_object = datetime.fromtimestamp(target_unix, tz=timezone.utc)
        date_str = dt_object.strftime('%Y-%m')
        
        params = {'appid': appid, 'count': 1, 'enddate': target_unix}
        
        try:
            time.sleep(0.5)
            response = requests.get(base_url, params=params)
            data = response.json()
            current_total_count = data['appnews']['count']
            
            if current_total_count == 0:
                break

            if last_total_count is not None:
                monthly_delta = last_total_count - current_total_count
                mapped_data.append({
                    "Date": date_str,
                    "Raw_Velocity": monthly_delta
                })
                print(f"{date_str} | Velocity: {monthly_delta}")

            last_total_count = current_total_count
            
        except Exception as e:
            print(f"Error at {date_str}: {e}")
            break
            
    return mapped_data

2026-05 | Velocity: 2
2026-04 | Velocity: 14
2026-03 | Velocity: 10
2026-02 | Velocity: 4
2026-01 | Velocity: 5
2025-12 | Velocity: 7
2025-11 | Velocity: 6
2025-10 | Velocity: 7
2025-09 | Velocity: 14
2025-08 | Velocity: 6
2025-07 | Velocity: 7
2025-06 | Velocity: 5
2025-05 | Velocity: 11
2025-04 | Velocity: 0
2025-03 | Velocity: 5
2025-02 | Velocity: 3
2025-01 | Velocity: 13
2024-12 | Velocity: 5
2024-11 | Velocity: 10
2024-10 | Velocity: 9
2024-09 | Velocity: 7
2024-08 | Velocity: 7
2024-07 | Velocity: 4
2024-06 | Velocity: 9
2024-05 | Velocity: 7
2024-04 | Velocity: 10
2024-03 | Velocity: 8
2024-02 | Velocity: 5
2024-01 | Velocity: 6
2023-12 | Velocity: 5
2023-11 | Velocity: 8
2023-10 | Velocity: 9
2023-09 | Velocity: 23
2023-08 | Velocity: 13
2023-07 | Velocity: 18
2023-06 | Velocity: 8
2023-05 | Velocity: 15
2023-04 | Velocity: 28
2023-03 | Velocity: 18
2023-02 | Velocity: 23
2023-01 | Velocity: 9
2022-12 | Velocity: 0
2022-11 | Velocity: 5
2022-10 | Velocity: 14
2022-09 | Velocit

,Date,Raw_Velocity
0,2026-05,2
1,2026-04,14
2,2026-03,10
3,2026-02,4
4,2026-01,5
...,...,...
155,2013-08,7
156,2013-07,3
157,2013-06,9
158,2013-05,12


In [ ]:
results = []
lookup_df = pd.read_csv("complete_steam_lookup_2026.csv")
with open('game_list.csv', mode = 'r', newline = '') as file:
    reader = csv.reader(file)
    header = next(reader)
    for row in reader:
        app_id = find_appid(row[0], lookup_df)
        df_details = get_updates(app_id)
        df = pd.DataFrame(df_details)
        if not df.empty and 'Date' in df.columns:
            df['date'] = pd.to_datetime(df['date'], unit = 's')
            df['Game'] = row[0]
            results.append(df)
            print(df)
        else:
            print(row[0])
        time.sleep(random.uniform(2, 5))
        
update_df = pd.concat(results, axis = 0, ignore_index = True)
update_df

2026-05 | Velocity: 2
2026-04 | Velocity: 14
2026-03 | Velocity: 10
2026-02 | Velocity: 4
2026-01 | Velocity: 5
2025-12 | Velocity: 7
2025-11 | Velocity: 6
2025-10 | Velocity: 7
2025-09 | Velocity: 14
2025-08 | Velocity: 6
2025-07 | Velocity: 7
2025-06 | Velocity: 5
2025-05 | Velocity: 11
2025-04 | Velocity: 0
2025-03 | Velocity: 5
2025-02 | Velocity: 3
2025-01 | Velocity: 13
2024-12 | Velocity: 5
2024-11 | Velocity: 10
2024-10 | Velocity: 9
2024-09 | Velocity: 7
2024-08 | Velocity: 7
2024-07 | Velocity: 4
2024-06 | Velocity: 9
2024-05 | Velocity: 7
2024-04 | Velocity: 10
2024-03 | Velocity: 8
2024-02 | Velocity: 5
2024-01 | Velocity: 6
2023-12 | Velocity: 5
2023-11 | Velocity: 8
2023-10 | Velocity: 9


KeyboardInterrupt: 